Prepare Data

In [1]:
!uv add requests

Resolved 113 packages in 18ms
Checked 109 packages in 8ms


In [18]:
import requests

repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)

In [19]:
len(zip_response.content)

17544837

In [20]:
import io
import zipfile

zip_arv = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [21]:
filesname = zip_arv.namelist()
len(filesname)
fi = (filesname[26:27][0])
#fi = fi.split('/',maxsplit=1)[-1]
fie_content = zip_arv.open(fi)
mr_conent = fie_content.read().decode('utf8')

In [22]:
print(mr_conent)

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.
</Check>

![](/images/alerts.png)

To enable alerts, open the Project and navigate to the "Alerts" in the left menu. You must set:

* A notification channel.

* An alert condition.

## Notification channels

You can choose between the following options:

* **Email**. Add email addresses to send alerts to.

* **Slack**. Add a Slack webhook.

* **Discord**. Add a Discord webhook.

## Alert conditions

### Failed tests

If you use Tests (conditional checks) in your Project, you can tie alerting to the failed Tests in a Test Suite. Toggle this option on the Alerts page. Evidently will set an alert to the defined channel if any of the Tests fail.

<Tip>
  **How to avoid alert fatigue?** Use the `is_critical` parameter to mark non-critical Test as Warnings. Setting it to `False` prevent alerts for those checks even if th

In [23]:
!uv add python-frontmatter
import frontmatter

Resolved 113 packages in 1ms
Checked 109 packages in 2ms


In [24]:
post = frontmatter.loads(mr_conent)
print(post.content[:100])

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [25]:
print(post.metadata)

{'title': 'Alerts', 'description': 'How to set up alerts.'}


In [26]:
filename_corrected = fi.split('/', 1)[-1]
print(filename_corrected)

docs/platform/alerts.mdx


In [27]:
doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

In [28]:
doc

{'content': '<Check>\n  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n![](/images/alerts.png)\n\nTo enable alerts, open the Project and navigate to the "Alerts" in the left menu. You must set:\n\n* A notification channel.\n\n* An alert condition.\n\n## Notification channels\n\nYou can choose between the following options:\n\n* **Email**. Add email addresses to send alerts to.\n\n* **Slack**. Add a Slack webhook.\n\n* **Discord**. Add a Discord webhook.\n\n## Alert conditions\n\n### Failed tests\n\nIf you use Tests (conditional checks) in your Project, you can tie alerting to the failed Tests in a Test Suite. Toggle this option on the Alerts page. Evidently will set an alert to the defined channel if any of the Tests fail.\n\n<Tip>\n  **How to avoid alert fatigue?** Use the `is_critical` parameter to mark non-critical Test as Warnings. Setting it to `False` prevent alerts for those checks even if they fail.\n</Tip>\n\n

In [29]:
def read_github_repository(repo_owner, repo_name, branch="main"):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response = requests.get(url)
    response.raise_for_status()

    documents = []
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md', '.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc = {
                    'content': post.content,
                    'title': post.metadata.get('title'),
                    'description': post.metadata.get('description'),
                    'filename': file_path.split('/', 1)[-1]
                }
                documents.append(doc)

    return documents

In [30]:
repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")

Downloaded 95 documents


In [31]:
!uv add gitsource

Resolved 113 packages in 1ms
Checked 109 packages in 1ms


In [32]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [33]:
documents = [f.parse() for f in files]
len(documents)

95

Search

In [37]:
query = 'LLM as a Judge'

In [34]:
!uv add minsearch

Resolved 123 packages in 2.32s                                       
Prepared 11 packages in 2.13s                                                scipy                ------------------------------ 32.84 MiB/33.69 MiB         scipy                ------------------------------ 16.76 MiB/33.69 MiB         scipy                ------------------------------ 15.75 MiB/33.69 MiB         scipy                ------------------------------ 10.56 MiB/33.69 MiB         scipy                ------------------------------ 717.91 KiB/33.69 MiB        scipy                ------------------------------ 589.91 KiB/33.69 MiB        scipy                ------------------------------ 189.91 KiB/33.69 MiB        
Uninstalled 1 package in 150ms
░░░░░░░░░░░░░░░░░░░░ [0/11] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardl

In [36]:
from minsearch import Index

In [38]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [39]:
results = index.search(query, num_results=5)

In [40]:
print(results)

[{'title': 'LLM as a judge', 'description': 'How to create and evaluate an LLM judge.', 'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to create 

Home work section